# M2M-100 418M + LoRA **v9** — 3 ngôn ngữ, 4 chiều: ja↔vi, en↔vi

**Nền tảng (từ v7/v8, giữ nguyên):** train từ M2M-100 gốc cả 4 chiều một lần, r=32, `skip_ja` cho 24 câu
`general` lệch nghĩa, eval 103 câu cố định, contrast set, chọn checkpoint theo eval nhiễu.

**Khác v8 — sửa chỗ LỆCH giữa lúc train và lúc app chạy thật (đối chiếu `backend/main.py`, `postprocess.py`):**
1. **Số bị phá khi làm nhiễu.** `noise_ja`/`noise_vi` v8 xoá `.` `,` cả trong số: `バージョン2.3` → `23`,
   `12.000` → `12 000`. Backend (`normalize_source`) lại đưa vào model `12.000` / `12,000` → v8 học sai.
   v9 giữ dấu nằm giữa hai chữ số.
2. **Số bằng chữ.** Zipformer-VI đọc số bằng chữ, `normalize_source` chỉ đổi số ≥ 100 → model thật nhận
   `ba giờ chiều`, `mười lăm phần trăm`, trong khi 92 câu train v8 toàn chữ số. v9: câu nguồn vi ở mức nhiễu
   được đổi số < 100 sang chữ (xác suất 0.9 × level). JA: thỉnh thoảng số nhỏ thành kanji (`三時`).
3. **Tên người.** `NameProtector` thay `<tên>さん` bằng placeholder `PnA`, còn people.json đổi tên sang
   `Chiba-san` — data v8 không có câu nào dạng này. v9: 30 câu tên người + 50% số câu ja→vi có tên được đổi sang
   `PnA/PnB` (model phải chép nguyên placeholder).
4. **VAD gộp câu.** Một segment hay chứa 2 câu không dấu câu. v9 thêm 6% dòng ghép 2 câu.
5. **Data mới** (`data_v9.jsonl` = v8 1.183 + **108**): `target_names` 30, `target_short` 36 (câu ngắn/đệm —
   loại hay bị bịa nghĩa), `target_jargon` 42 (工数, 切り分け, 横展開, 本番反映, 差分, 納期, 持ち帰り, 影響範囲, 手戻り…
   — v8 có 0 câu). Sinh bằng `new_pairs_v9.py`.

**Sửa sau review (trước lần train đầu):**
6. **Regex tên hỏng.** `\b[A-Z][a-z]+-san\b` — kana/kanji cũng là `\w` nên `Ota-sanは` không có ranh giới từ →
   sau nhiễu khớp **0/30** câu, mục 3 coi như không chạy. Đổi sang lookaround ASCII → 30/30.
7. **Nhiễu sinh 1 lần cho cả 7 epoch.** Giờ sinh `NOISE_COPIES = 3` bản nhiễu khác seed, train 2 epoch
   (= 6 lượt nhìn dữ liệu, 3 biến thể nhiễu), eval/lưu checkpoint sau mỗi 1/3 epoch.
8. **Số kèm đơn vị** bám `normalize_source`: `5 triệu` → `5.000.000`, `1,5 triệu` → `1.500.000`, `2,5%` → `2,5 phần trăm`
   (trước đây thành `năm triệu` — app không bao giờ đưa dạng đó vào model).
9. **Đại từ.** Data gọi người nghe gần như toàn `anh` → 30% câu NGUỒN vi đổi `anh` → `chị` (chỉ ở nguồn).
10. **Eval sát app:** cột APP giờ decode y hệt backend (`repetition_penalty` 1.1, trần độ dài theo `LEN_MULT`).
    Cột SẠCH/NHIỄU giữ decode cũ để còn so với v6–v8.
11. **Bộ CHUNG** (CELL 9b): 40 câu đời thường ngoài domain IT — đo LoRA có làm model quên câu thường không.

**Eval:** 3 bộ — `SẠCH`, `NHIỄU` (nhiễu kiểu v8, **so thẳng được** với v6/v7/v8), `APP` (nhiễu v9 sát app,
dùng để chọn checkpoint) + bộ CHUNG ngoài domain. Contrast set thêm 11 case v9.

**Cách chạy:** Runtime → Change runtime type → **T4 GPU**. Upload `data_v9.jsonl` vào `/content`. **Run all** (~25 phút).
Điền số v8 vào `V8 = {...}` ở CELL 9 nếu đã chạy v8 (để trống thì bỏ qua cột đó).

**Kết quả:** `m2m100_418M_int8_v9.zip` → giải nén vào `backend/models/m2m100_418M_int8_v9/` rồi A/B ở máy local.

In [ ]:
# CELL 1 — cài đặt
!pip install -q "transformers==5.14.1" "peft==0.20.0" datasets accelerate sentencepiece sacrebleu
!pip uninstall -y torchao -q

import torch, transformers, peft
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ KHÔNG CÓ GPU")
assert torch.cuda.is_available(), "Bật GPU: Runtime -> Change runtime type -> T4 GPU"

# ⚠️ Backend (venv local) đang chạy transformers 5.14.1. Train và serve cùng một
# bản để tokenizer M2M-100 hành xử giống hệt nhau.

In [ ]:
# CELL 2 — nạp dữ liệu
import json, random, re, unicodedata, collections
from pathlib import Path

DATA_FILE = "data_v9.jsonl"
# Để file trên Drive thì bỏ comment 2 dòng dưới:
# from google.colab import drive; drive.mount("/content/drive")
# DATA_FILE = "/content/drive/MyDrive/<thư_mục>/data_v9.jsonl"

LANGS = ("vi", "ja", "en")
DIRECTIONS = [("ja", "vi"), ("vi", "ja"), ("en", "vi"), ("vi", "en")]

ALL = []
for i, line in enumerate(open(DATA_FILE, encoding="utf-8"), 1):
    line = line.strip()
    if not line:
        continue
    d = json.loads(line)
    missing = [k for k in (*LANGS, "id", "origin", "split") if not d.get(k)]
    if missing:
        raise ValueError(f"Dòng {i} thiếu {missing}: {line[:80]}")
    ALL.append({k: (v.strip() if isinstance(v, str) else v) for k, v in d.items()})

eval_rows  = [r for r in ALL if r["split"] == "eval"]
train_rows = [r for r in ALL if r["split"] == "train"]

# Không câu nguồn nào của eval được lọt vào train (ở bất kỳ ngôn ngữ nào)
for lang in LANGS:
    leak = {r[lang] for r in eval_rows} & {r[lang] for r in train_rows}
    assert not leak, f"Rò rỉ eval->train ({lang}): {list(leak)[:3]}"

print(f"Tổng {len(ALL)} bộ ba | train {len(train_rows)} | eval {len(eval_rows)}")
for o, n in sorted(collections.Counter(r["origin"] for r in train_rows).items()):
    print(f"  train/{o:16s} {n:4d}")
n_skip = sum(1 for r in train_rows if r.get("skip_ja"))
print(f"  skip_ja (vi↔ja lệch nghĩa, chỉ dùng cho en↔vi): {n_skip}")
assert not any(r.get("skip_ja") for r in eval_rows), "Eval không được có câu lệch nghĩa"
assert len(eval_rows) == 103, "Eval phải đúng 103 câu của v6 để so sánh được"
print(f"  câu có tên X-san: {sum(1 for r in train_rows if re.search(r'[A-Z][a-z]+-san', r['ja']))}")

In [ ]:
# CELL 3 — mô phỏng lỗi ASR cho câu NGUỒN (câu đích luôn sạch)
#
# v9 bám ĐÚNG thứ backend đưa vào model (main.py -> postprocess.prepare_source):
#  - Số: normalize_source xuất "12.000" (vi) / "12,000" (ja), ASR giữ "2.3" -> KHÔNG được
#    xoá dấu nằm giữa hai chữ số (v8 xoá -> dạy model "23" nghĩa là "2.3").
#  - Zipformer-VI đọc số bằng CHỮ; normalize_source chỉ đổi số >= 100 -> số < 100 tới
#    model dạng "ba giờ", "mười lăm phần trăm".
#  - Tên trong people.json tới model dạng "Chiba-san" (ja) / "chiba-san" (vi) -> giữ "-".
# Bản v8 giữ lại (noise_*_v8) CHỈ để dựng cột "NHIỄU" ở CELL 9, so thẳng với v6/v7/v8.

# ---- JA. Parakeet-JA (NeMo CTC) không sinh dấu câu, không khoảng trắng.
# ー và ・ là một phần của từ -> KHÔNG xoá (bug của v4).
JA_PUNCT_V8 = re.compile(r"[。、！？!?,.「」『』（）()…―〜]")
JA_PUNCT    = re.compile(r"[。、！？!?「」『』（）()…―〜]|(?<!\d)[,.]|[,.](?!\d)")
_KAN = "〇一二三四五六七八九"

def _kanji_small(n):                                  # 1..99 -> 一..九十九
    t, o = divmod(n, 10)
    return ((_KAN[t] if t > 1 else "") + "十" if t else "") + (_KAN[o] if o else "")

def noise_ja_v8(s, rng=None, level=1.0):
    s = JA_PUNCT_V8.sub("", s)
    return re.sub(r"[\s　]+", "", s)

def noise_ja(s, rng=None, level=1.0):
    s = JA_PUNCT.sub("", s)
    s = re.sub(r"[\s　]+", "", s)
    if level and rng is not None and rng.random() < 0.25 * level:
        # ASR đôi khi ra kanji cho số nhỏ đi kèm đơn vị; normalize_source để nguyên số < 100
        s = re.sub(r"(?<![\d.,])([1-9]\d?)(?=[時分人日回件つ個台歳])",
                   lambda m: _kanji_small(int(m.group(1))), s)
    return s

# ---- VI. Zipformer-VI: không dấu câu, chữ thường, số bằng chữ, sai dấu thanh, nhầm phụ âm, nuốt từ.
TONES   = "̣̀́̃̉"
ASCII_W = re.compile(r"^[a-z0-9._/#-]+$")
VN_CONFUS = [("d","gi"),("gi","d"),("ch","tr"),("tr","ch"),("s","x"),("x","s"),("n","l"),("l","n")]

def _corrupt_tone(w, rng):
    d = list(unicodedata.normalize("NFD", w))
    idx = [i for i, c in enumerate(d) if c in TONES]
    if not idx:
        return w
    i = rng.choice(idx)
    d[i] = "" if rng.random() < 0.45 else rng.choice([t for t in TONES if t != d[i]])
    return unicodedata.normalize("NFC", "".join(d))

def _vi_word_noise(s, rng, level):
    words, out = s.split(), []
    for w in words:
        if ASCII_W.match(w):
            out.append(w); continue
        r = rng.random()
        if r < 0.04 * level and len(words) > 6:
            continue
        elif r < 0.24 * level:
            w = _corrupt_tone(w, rng)
        elif r < 0.30 * level:
            for a, b in VN_CONFUS:
                if w.startswith(a):
                    w = b + w[len(a):]; break
        out.append(w)
    return " ".join(out)

def noise_vi_v8(s, rng=None, level=1.0):
    s = unicodedata.normalize("NFC", s)
    s = re.sub(r"[.,!?;:\"'’“”()\[\]\-–—…]", " ", s)
    s = re.sub(r"\s+", " ", s).strip().lower()
    if not level or rng is None:
        return s
    return _vi_word_noise(s, rng, level)

_VI_UNITS = ["không", "một", "hai", "ba", "bốn", "năm", "sáu", "bảy", "tám", "chín"]

def vi_num_words(n):
    """0..99 theo cách Zipformer đọc (và postprocess._VI_DIGIT hiểu): 21 -> hai mươi mốt."""
    if n < 10:
        return _VI_UNITS[n]
    t, o = divmod(n, 10)
    head = "mười" if t == 1 else _VI_UNITS[t] + " mươi"
    if o == 0:
        return head
    tail = {1: "mốt" if t > 1 else "một", 4: "tư" if t > 1 else "bốn", 5: "lăm"}.get(o, _VI_UNITS[o])
    return f"{head} {tail}"

VI_PCT      = re.compile(r"(?<![\w.,])(\d{1,2})\s?%")
VI_PCT_REST = re.compile(r"(\d[\d.,]*)\s?%")
VI_INT      = re.compile(r"(?<![\w.,:/])(\d{1,2})(?!\w|[.,:/]\d)")
# normalize_source đổi "năm triệu" -> "5.000.000", "một phẩy năm triệu" -> "1.500.000"
# -> model KHÔNG BAO GIỜ thấy "năm triệu"; trước bản sửa này noise_vi lại sinh đúng dạng đó.
VI_SCALE_NUM = re.compile(r"(?<![\w.,])(\d{1,3})(?:,(\d{1,2}))?\s+(nghìn|ngàn|triệu|tỷ|tỉ)(?!\w)")
_VI_MUL = {"nghìn": 10**3, "ngàn": 10**3, "triệu": 10**6, "tỷ": 10**9, "tỉ": 10**9}

def _vi_scale(m):
    whole, frac, mul = int(m.group(1)), m.group(2) or "", _VI_MUL[m.group(3)]
    n = whole * mul + (int(frac) * mul // 10 ** len(frac) if frac else 0)
    return f"{n:,}".replace(",", ".")

def vi_numbers_as_asr(s):
    s = VI_SCALE_NUM.sub(_vi_scale, s)
    s = VI_PCT.sub(lambda m: vi_num_words(int(m.group(1))) + " phần trăm", s)
    s = VI_PCT_REST.sub(r"\1 phần trăm", s)       # "2,5%" -> "2,5 phần trăm" (app đọc "hai phẩy năm phần trăm")
    return VI_INT.sub(lambda m: vi_num_words(int(m.group(1))), s)

# Dấu câu bỏ đi; "." "," giữa hai chữ số và "-" giữa hai chữ/số (Chiba-san, kick-off) được giữ.
VI_PUNCT = re.compile(r"[!?;:\"'’“”()\[\]–—…]|(?<!\d)[.,]|[.,](?!\d)|(?<![A-Za-z0-9])-|-(?![A-Za-z0-9])")

def noise_vi(s, rng=None, level=1.0):
    s = unicodedata.normalize("NFC", s)
    if level and rng is not None and rng.random() < 0.9 * level:
        s = vi_numbers_as_asr(s)
    s = VI_PUNCT.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip().lower()
    if not level or rng is None:
        return s
    return _vi_word_noise(s, rng, level)

# ---- EN: giữ nguyên v7/v8. Parakeet-EN (TDT 0.6B v2) xuất câu gần như sạch: có dấu câu,
# viết hoa, chữ số. Lỗi định dạng hay gặp nhất là "4:30" -> "4.30". Nhiễu NHẸ.
def noise_en(s, rng=None, level=1.0):
    s = unicodedata.normalize("NFC", s).strip()
    if not level or rng is None:
        return s
    if rng.random() < 0.5 * level:
        s = re.sub(r"\b(\d{1,2}):(\d{2})\b", r"\1.\2", s)     # 4:30 -> 4.30
    if rng.random() < 0.25 * level:
        s = s.replace(",", "")                               # mất dấu phẩy
    if rng.random() < 0.25 * level:
        s = s.rstrip(".!?")                                  # mất dấu chấm cuối
    if rng.random() < 0.10 * level:
        s = s.lower()
    words = s.split()
    if len(words) > 8 and rng.random() < 0.08 * level:       # nuốt một từ ngắn
        cand = [i for i, w in enumerate(words) if len(w) <= 3]
        if cand:
            del words[rng.choice(cand)]
            s = " ".join(words)
    return s

NOISE    = {"ja": noise_ja,    "vi": noise_vi,    "en": noise_en}
NOISE_V8 = {"ja": noise_ja_v8, "vi": noise_vi_v8, "en": noise_en}

# Tự kiểm tra các lỗi v8 đã sửa
assert noise_ja("バージョン2.3です。") == "バージョン2.3です"
assert noise_ja("ユーザー数は12,000人です。") == "ユーザー数は12,000人です"
assert noise_vi("Hệ thống có 12.000 người dùng.") == "hệ thống có 12.000 người dùng"
assert noise_vi("Họp lúc 3 giờ, tăng 15%.", random.Random(0), 0.0) == "họp lúc 3 giờ tăng 15%"
assert vi_numbers_as_asr("Tăng 15% lúc 3 giờ, 21 người, version 2.3") == \
       "Tăng mười lăm phần trăm lúc ba giờ, hai mươi mốt người, version 2.3"
assert noise_vi("Chiba-san xem giúp em.") == "chiba-san xem giúp em"
assert vi_numbers_as_asr("Ngân sách là 5 triệu yên, tỷ lệ 2,5%") == \
       "Ngân sách là 5.000.000 yên, tỷ lệ 2,5 phần trăm"
assert vi_numbers_as_asr("khoảng 1,5 triệu, hơn 10 nghìn người") == "khoảng 1.500.000, hơn 10.000 người"

_r = random.Random(3)
demo = [x for x in train_rows if x["origin"] in ("target_time", "target_num", "target_names")]
for r in demo[:2] + demo[45:47] + demo[-2:]:
    print(f"vi gốc : {r['vi']}\n   nhiễu: {noise_vi(r['vi'], _r, 1.0)}")
    print(f"ja gốc : {r['ja']}\n   nhiễu: {noise_ja(r['ja'], _r, 1.0)}\n")

In [ ]:
# CELL 4 — bộ ba -> dòng train/eval theo 4 chiều
from datasets import Dataset

# Lặp lại dữ liệu targeted bao nhiêu lần. 1 = không lặp. Tăng lên 2 nếu contrast
# set (CELL 11) vẫn trượt nhiều case sau khi train.
TARGET_REPEAT = 1
NAME_SLOT_P   = 0.5    # tỉ lệ câu nguồn ja có tên được đổi sang placeholder PnA/PnB
CONCAT_P      = 0.06   # số dòng ghép 2 câu, tính theo % số dòng train
NOISE_COPIES  = 3      # số bản nhiễu KHÁC NHAU của mỗi dòng (bản đầu v9: 1 bản dùng cho cả 7 epoch)
PRONOUN_P     = 0.3    # tỉ lệ câu nguồn vi có "anh" (người nghe) được đổi sang "chị"

# KHÔNG dùng \b: kana/kanji cũng là \w -> "Ota-sanは" không có ranh giới từ sau "san"
# (bản đầu v9 khớp 0/30 câu sau nhiễu).
NAME_RE = re.compile(r"(?<![A-Za-z])[A-Z][a-z]+-san(?![A-Za-z])")

# "anh" chỉ người nghe; bỏ qua "anh ấy/anh ta" (ngôi 3), "anh em", "anh chị", "tiếng Anh" và
# "Anh" viết hoa giữa câu (tên người: "Tuấn Anh").
_ANH_TAIL  = r"(?!\s+(?:ấy|ta|em|chị|ruột|trai|họ)\b)"
_ANH_LOWER = re.compile(r"(?<![Tt]iếng )\banh\b" + _ANH_TAIL)
_ANH_START = re.compile(r"^Anh\b" + _ANH_TAIL)
_MALE      = re.compile(r"\b(he|him|his|brother)\b|彼|兄", re.I)

def swap_pronoun(r, rng):
    """Người nghe trong data gần như toàn "anh" -> model ít gặp "chị". Chỉ đổi ở NGUỒN vi:
    câu đích vi thì không đổi được vì app không biết giới tính người nghe."""
    s = r["vi"]
    if rng is None or rng.random() >= PRONOUN_P or _MALE.search(r["en"] + r["ja"]):
        return s
    return _ANH_LOWER.sub("chị", _ANH_START.sub("Chị", s))

def slot_names(src, tgt, rng):
    """Giả lập NameProtector (postprocess.py): '<tên>さん' -> ' PnA ' ở câu nguồn ja;
    model phải chép nguyên PnA sang câu đích thì backend mới bung lại được tên."""
    names = [n for n in dict.fromkeys(NAME_RE.findall(src)) if n in tgt]
    if not names or rng is None or rng.random() >= NAME_SLOT_P:
        return src, tgt
    for i, n in enumerate(names[:8]):
        slot = "Pn" + chr(ord("A") + i)
        src, tgt = src.replace(n, f" {slot} "), tgt.replace(n, slot)
    return re.sub(r"\s+", " ", src).strip(), tgt

def to_rows(rows, rng=None, levels=(0.0,), noise=None, augment=False):
    noise = noise or NOISE
    out = []
    for r in rows:
        for s, t in DIRECTIONS:
            # skip_ja: cặp vi↔ja gốc lệch nghĩa (phụ đề ghép tự động). Bản en dịch
            # theo vế vi nên vẫn dùng cho en↔vi, chỉ bỏ các chiều có tiếng Nhật.
            if r.get("skip_ja") and "ja" in (s, t):
                continue
            lv = rng.choice(levels) if rng else levels[0]
            raw = swap_pronoun(r, rng) if augment and s == "vi" else r[s]
            src, tgt = noise[s](raw, rng, lv), r[t]
            if s == "ja":                      # backend chỉ đóng băng tên ở nguồn ja
                src, tgt = slot_names(src, tgt, rng)
            out.append({"src_text": src, "tgt_text": tgt,
                        "src_lang": s, "tgt_lang": t, "level": lv, "origin": r["origin"]})
    return out

def concat_rows(rows, n, rng, levels=(0.5, 1.0)):
    """VAD hay gộp 2 câu liền nhau thành 1 segment, ASR không có dấu câu ở giữa."""
    pool = [r for r in rows if len(r["vi"]) < 90]
    out = []
    while len(out) < n:
        a, b = rng.sample(pool, 2)
        s, t = rng.choice(DIRECTIONS)
        if (a.get("skip_ja") or b.get("skip_ja")) and "ja" in (s, t):
            continue
        lv = rng.choice(levels)
        src = ("" if s == "ja" else " ").join(NOISE[s](x[s], rng, lv) for x in (a, b))
        tgt = ("" if t == "ja" else " ").join(x[t] for x in (a, b))
        if s == "ja":
            src, tgt = slot_names(src, tgt, rng)
        out.append({"src_text": src, "tgt_text": tgt, "src_lang": s, "tgt_lang": t,
                    "level": lv, "origin": "concat"})
    return out

base = train_rows + [r for r in train_rows if r["origin"].startswith("target_")] * (TARGET_REPEAT - 1)
# Mỗi bản dùng seed riêng -> cùng một câu có NOISE_COPIES biến thể nhiễu / mức nhiễu khác nhau.
train_list = []
for c in range(NOISE_COPIES):
    train_list += to_rows(base, random.Random(c), levels=(0.0, 0.5, 1.0), augment=True)  # sạch / yên / ồn
rng = random.Random(100)
train_list += concat_rows(train_rows, int(len(train_list) * CONCAT_P), rng)
random.Random(1).shuffle(train_list)

train_dataset    = Dataset.from_list(train_list)
eval_dataset_raw = Dataset.from_list(to_rows(eval_rows, None, levels=(0.0,)))
# NHIỄU: y hệt v6/v7/v8 (cùng seed, cùng hàm nhiễu cũ) -> số so thẳng được với các bản trước
eval_noisy_raw   = Dataset.from_list(to_rows(eval_rows, random.Random(99), levels=(1.0,), noise=NOISE_V8))
# APP: nhiễu v9, sát thứ backend thật đưa vào model -> dùng chọn checkpoint
eval_app_raw     = Dataset.from_list(to_rows(eval_rows, random.Random(99), levels=(1.0,)))

print(f"📦 Train {len(train_dataset)} dòng | Eval sạch {len(eval_dataset_raw)} | "
      f"Eval nhiễu {len(eval_noisy_raw)} | Eval app {len(eval_app_raw)}")
print("Theo chiều:", dict(collections.Counter(f"{r['src_lang']}->{r['tgt_lang']}" for r in train_list)))
print("Theo mức nhiễu:", dict(collections.Counter(r["level"] for r in train_list)))
print("Dòng ghép 2 câu:", sum(1 for r in train_list if r["origin"] == "concat"),
      "| dòng có placeholder PnA:", sum(1 for r in train_list if "PnA" in r["src_text"]),
      "| nguồn vi có 'chị':", sum(1 for r in train_list if r["src_lang"] == "vi"
                                  and re.search(r"\bchị\b", r["src_text"])))
n_pna = sum(1 for r in train_list if "PnA" in r["src_text"])
assert n_pna >= 10, f"Chỉ {n_pna} dòng có PnA — slot_names không chạy (kiểm tra NAME_RE)"
for r in [x for x in train_list if "PnA" in x["src_text"]][:2] + \
         [x for x in train_list if x["origin"] == "concat"][:2]:
    print(f"  [{r['src_lang']}→{r['tgt_lang']}] {r['src_text']}\n      → {r['tgt_text']}")

In [ ]:
# CELL 5 — model + LoRA
import torch
from transformers import (M2M100ForConditionalGeneration, M2M100Tokenizer,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer,
                          DataCollatorForSeq2Seq, EarlyStoppingCallback)
from peft import LoraConfig, get_peft_model, TaskType

model_id = "facebook/m2m100_418M"
tokenizer = M2M100Tokenizer.from_pretrained(model_id)
model = M2M100ForConditionalGeneration.from_pretrained(model_id)

# v6: r=24. v7 học gấp đôi số chiều (thêm en↔vi) -> tăng dung lượng adapter lên r=32.
# Adapter vẫn chỉ ~3% tham số; model sau merge có kích thước y như cũ.
lora_config = LoraConfig(
    r=32, lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"],
    lora_dropout=0.1, bias="none", task_type=TaskType.SEQ_2_SEQ_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

MAX_LEN = 128

def preprocess(ex):
    # M2M-100: set src_lang/tgt_lang TRƯỚC khi encode (chèn token ngôn ngữ)
    tokenizer.src_lang = ex["src_lang"]
    mi = tokenizer(ex["src_text"], max_length=MAX_LEN, truncation=True)
    tokenizer.tgt_lang = ex["tgt_lang"]
    mi["labels"] = tokenizer(text_target=ex["tgt_text"], max_length=MAX_LEN,
                             truncation=True)["input_ids"]
    return mi

tokenized_train      = train_dataset.map(preprocess,    remove_columns=train_dataset.column_names)
tokenized_eval       = eval_dataset_raw.map(preprocess, remove_columns=eval_dataset_raw.column_names)
tokenized_eval_noisy = eval_noisy_raw.map(preprocess,   remove_columns=eval_noisy_raw.column_names)
tokenized_eval_app   = eval_app_raw.map(preprocess,     remove_columns=eval_app_raw.column_names)

# PeftModel bọc ngoài làm DataCollatorForSeq2Seq không thấy
# prepare_decoder_input_ids_from_labels -> đưa base model cho collator.
collator_model = model.get_base_model() if hasattr(model, "get_base_model") else model
data_collator = DataCollatorForSeq2Seq(tokenizer, model=collator_model)
print(f"Train {len(tokenized_train)} | Eval sạch {len(tokenized_eval)} | Eval nhiễu {len(tokenized_eval_noisy)} | Eval app {len(tokenized_eval_app)}")

In [ ]:
# CELL 6 — kiểm tra forward/backward trước khi train
from torch.utils.data import DataLoader

dev = "cuda"
model.to(dev)
batch = next(iter(DataLoader(tokenized_train.select(range(4)), batch_size=2, collate_fn=data_collator)))
print("Khoá trong batch:", {k: tuple(v.shape) for k, v in batch.items()})
model.train()
out = model(**{k: v.to(dev) for k, v in batch.items()})
out.loss.backward(); model.zero_grad(set_to_none=True)
print(f"✅ forward + backward OK — loss = {out.loss.item():.4f}")

In [ ]:
# CELL 7 — train
# 1 epoch giờ = NOISE_COPIES bản nhiễu -> to gấp 3 epoch v8. EPOCHS = 2 ~ 6 epoch kiểu v8
# (v7 tốt nhất ở epoch 4). Eval + lưu sau mỗi 1/NOISE_COPIES epoch (= 1 epoch kiểu v8) để
# early stopping vẫn chọn được mốc mịn như trước.
EPOCHS = 2
BS = 16             # T4 dư sức với 418M + LoRA ở MAX_LEN 128
steps_per_epoch = -(-len(tokenized_train) // BS)
eval_steps = -(-steps_per_epoch // NOISE_COPIES)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(10, int(total_steps * 0.06))
print(f"steps/epoch {steps_per_epoch} | eval mỗi {eval_steps} step | tối đa {total_steps} | warmup {warmup_steps}")

args = Seq2SeqTrainingArguments(
    output_dir="./m2m418-lora-v9",
    learning_rate=3e-4,              # v6: 2.5e-4 @ BS 8. Batch gấp đôi -> nâng nhẹ
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    per_device_train_batch_size=BS,
    per_device_eval_batch_size=32,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    fp16=True,
    # KHÔNG bật label_smoothing_factor: Trainer pop 'labels' -> M2M100 không dựng
    # được decoder_input_ids -> ValueError (đã gặp ở v5).
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=eval_steps,              # load_best_model_at_end cần save_steps = bội của eval_steps
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42,
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval_app,        # v9: chọn checkpoint theo eval nhiễu SÁT APP, cả 4 chiều
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()

In [ ]:
# CELL 8 — lưu adapter + đường loss
import os, matplotlib.pyplot as plt, numpy as np

os.makedirs("./adapter_best_v9", exist_ok=True)
model.save_pretrained("./adapter_best_v9")
tokenizer.save_pretrained("./adapter_best_v9")
json.dump(trainer.state.log_history, open("./adapter_best_v9/log_history.json", "w"),
          ensure_ascii=False, indent=1)
print("✅ adapter_best_v9 đã lưu")
# Muốn giữ khi Colab reset:
# from google.colab import drive; drive.mount("/content/drive")
# import shutil; shutil.make_archive("/content/drive/MyDrive/m2m418_adapter_v9", "zip", "./adapter_best_v9")

h = trainer.state.log_history
tr = [(x["epoch"], x["loss"]) for x in h if "loss" in x and "eval_loss" not in x]
ev = [(x["epoch"], x["eval_loss"]) for x in h if "eval_loss" in x]
plt.figure(figsize=(7, 3.5))
if tr: plt.plot(*zip(*tr), label="train", alpha=.7)
if ev: plt.plot(*zip(*ev), label="eval (nhiễu)", marker="s", lw=2)
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=.3); plt.show()
for e, l in ev:
    print(f"  epoch {e:.2f}: {l:.4f}{'  ← best' if l == min(y for _, y in ev) else ''}")

In [ ]:
# CELL 9 — chrF cho 4 chiều: v9 vs M2M-100 GỐC (tắt adapter) vs mốc v6, v7, v8
import sacrebleu, numpy as np, torch

RESULTS = {}
LEN_MULT = {("ja", "vi"): 4.0, ("vi", "ja"): 2.0, ("en", "vi"): 3.0, ("vi", "en"): 2.0}   # = backend/main.py

@torch.no_grad()
def gen_app(text, src, tgt):
    """Decode y hệt backend translate(): vi viết thường, beam 4, repetition_penalty 1.1,
    trần độ dài min(128, n_src * LEN_MULT + 12). Từng câu một vì trần độ dài tính theo câu."""
    model.eval()
    tokenizer.src_lang = src
    enc = tokenizer([text.lower() if src == "vi" else text], return_tensors="pt").to(model.device)
    cap = min(128, int(enc["input_ids"].shape[1] * LEN_MULT[(src, tgt)]) + 12)
    o = model.generate(**enc, forced_bos_token_id=tokenizer.get_lang_id(tgt),
                       max_new_tokens=cap, num_beams=4, early_stopping=True,
                       no_repeat_ngram_size=3, repetition_penalty=1.1)
    return tokenizer.batch_decode(o, skip_special_tokens=True)[0].strip()

@torch.no_grad()
def gen_batch(texts, src, tgt, bs=32, app=False):
    # app=False: decode cũ của v6–v8 (giữ để cột SẠCH/NHIỄU còn so được với mốc cũ)
    if app:
        return [gen_app(t, src, tgt) for t in texts]
    outs = []; model.eval()
    for i in range(0, len(texts), bs):
        tokenizer.src_lang = src
        enc = tokenizer(texts[i:i+bs], return_tensors="pt", padding=True,
                        truncation=True, max_length=128).to(model.device)
        o = model.generate(**enc, forced_bos_token_id=tokenizer.get_lang_id(tgt),
                           max_length=128, num_beams=4, early_stopping=True,
                           no_repeat_ngram_size=3, repetition_penalty=1.05)
        outs += tokenizer.batch_decode(o, skip_special_tokens=True)
    return outs

def boot_ci(sent, n=1000, seed=0):
    r = np.random.default_rng(seed); a = np.asarray(sent)
    m = [a[r.integers(0, len(a), len(a))].mean() for _ in range(n)]
    return np.percentile(m, 2.5), np.percentile(m, 97.5)

def score(ds, label, tag="v9", app=False):
    print(f"\n===== {tag} / {label} =====")
    for src, tgt in DIRECTIONS:
        sub = [r for r in ds if r["src_lang"] == src and r["tgt_lang"] == tgt]
        srcs = [r["src_text"] for r in sub]; refs = [r["tgt_text"] for r in sub]
        preds = gen_batch(srcs, src, tgt, app=app)
        chrf = sacrebleu.corpus_chrf(preds, [refs]).score
        sent = [sacrebleu.sentence_chrf(p, [r]).score for p, r in zip(preds, refs)]
        lo, hi = boot_ci(sent)
        RESULTS[(tag, label, src, tgt)] = dict(chrf=chrf, sent=sent, srcs=srcs, refs=refs,
                                               preds=preds, ci=(lo, hi))
        print(f"  {src}→{tgt}: chrF {chrf:5.2f} | CI95 câu [{lo:5.1f}, {hi:5.1f}] (n={len(sub)})")

score(eval_dataset_raw, "SẠCH")
score(eval_noisy_raw,   "NHIỄU")
score(eval_app_raw,     "APP", app=True)        # APP: decode y hệt backend
with model.disable_adapter():                 # cùng model, tắt LoRA = M2M-100 gốc
    score(eval_noisy_raw, "NHIỄU", tag="gốc")
    score(eval_app_raw,   "APP",   tag="gốc", app=True)

# Mốc (cùng 103 câu eval, cột NHIỄU cùng hàm nhiễu v8). v6 chép từ notebook v6 (không có en);
# v7 chép từ lần chạy v7. V8: ĐIỀN số từ lần chạy v8 nếu có — để nan thì bỏ qua.
nan = float("nan")
V6 = {("NHIỄU", "ja", "vi"): 49.53, ("NHIỄU", "vi", "ja"): 38.92}
V7 = {("NHIỄU", "ja", "vi"): 51.63, ("NHIỄU", "vi", "ja"): 38.21,
      ("NHIỄU", "en", "vi"): 56.70, ("NHIỄU", "vi", "en"): 53.57,
      ("SẠCH", "ja", "vi"): 51.63, ("SẠCH", "vi", "ja"): 49.82,
      ("SẠCH", "en", "vi"): 56.28, ("SẠCH", "vi", "en"): 63.89}
V8 = {("NHIỄU", "ja", "vi"): nan, ("NHIỄU", "vi", "ja"): nan,
      ("NHIỄU", "en", "vi"): nan, ("NHIỄU", "vi", "en"): nan}

print("\n" + "=" * 96)
print(f"{'chiều':8s}{'gốc':>8s}{'v6':>8s}{'v7':>8s}{'v8':>8s}{'v9':>8s}{'v9 sạch':>9s}"
      f"{'APP gốc':>9s}{'APP v9':>8s}   đánh giá (cột nhiễu)")
print("=" * 96)
for src, tgt in DIRECTIONS:
    g  = RESULTS[("gốc", "NHIỄU", src, tgt)]["chrf"]
    n9 = RESULTS[("v9", "NHIỄU", src, tgt)]
    c9 = RESULTS[("v9", "SẠCH", src, tgt)]["chrf"]
    ag = RESULTS[("gốc", "APP", src, tgt)]["chrf"]
    a9 = RESULTS[("v9", "APP", src, tgt)]["chrf"]
    v6, v7, v8 = (V.get(("NHIỄU", src, tgt), nan) for V in (V6, V7, V8))
    half = (n9["ci"][1] - n9["ci"][0]) / 2
    # So với bản TỐT NHẤT trước đó; chênh trong nửa CI = ngang (103 câu không đọc được chênh nhỏ hơn)
    best = max(v for v in (v6, v7, v8) if v == v)
    d = n9["chrf"] - best
    verdict = ("✅ tốt hơn" if d > half else "✅ ngang" if d >= -half else f"🔴 tụt quá CI ({d:+.1f})")
    print(f"{src}→{tgt:5s}{g:8.2f}{v6:8.2f}{v7:8.2f}{v8:8.2f}{n9['chrf']:8.2f}{c9:9.2f}"
          f"{ag:9.2f}{a9:8.2f}   {verdict}")
print("\nv9 không được tụt quá CI so với bản tốt nhất trước đó ở cột NHIỄU (so thẳng được).")
print("Cột APP dùng nhiễu v9 (số bằng chữ, giữ dấu trong số) + decode y hệt backend — sát app thật nhất,")
print("nhưng chưa có mốc cũ. Cột SẠCH/NHIỄU giữ decode cũ (rep 1.05, max_length 128) để so với v6–v8.")

In [ ]:
# CELL 9b — bộ CHUNG ngoài domain IT: LoRA có làm model QUÊN câu đời thường không?
# 103 câu eval ở trên cùng nguồn với data train (IT + họp) nên không bắt được lỗi này.
# 40 câu dưới KHÔNG có trong data. So v9 với M2M-100 gốc, decode y hệt backend.
# n=40/chiều nên CI rộng — chỉ đọc được mức tụt lớn.
GENERAL = [
 ("Hôm nay trời mưa to quá, đường ngập hết rồi.", "今日は雨がすごくて、道路が冠水しています。", "It's raining so hard today that the streets are flooded."),
 ("Cuối tuần này gia đình tôi định đi biển Nha Trang.", "今週末、家族でニャチャンの海に行く予定です。", "My family is planning to go to the beach in Nha Trang this weekend."),
 ("Dạo này tôi hay bị mất ngủ nên hơi mệt.", "最近よく眠れなくて、少し疲れています。", "I haven't been sleeping well lately, so I'm a bit tired."),
 ("Quán phở gần nhà tôi rất ngon.", "家の近くのフォーの店はとてもおいしいです。", "The pho place near my house is really good."),
 ("Tết năm nay bạn có về quê không?", "今年のテトは実家に帰りますか。", "Are you going back to your hometown for Tet this year?"),
 ("Tháng trước con tôi vừa vào lớp một.", "先月、子どもが小学校に入学しました。", "My child started first grade last month."),
 ("Giá xăng lại tăng rồi, đi làm tốn kém hơn nhiều.", "ガソリン代がまた上がって、通勤にかなりお金がかかるようになりました。", "Gas prices went up again, so commuting costs a lot more now."),
 ("Tôi đang học nấu món Nhật vào buổi tối.", "夜に日本料理の作り方を習っています。", "I'm learning to cook Japanese food in the evenings."),
 ("Sân bay đông quá, tôi phải xếp hàng gần 1 tiếng.", "空港がとても混んでいて、1時間近く並びました。", "The airport was so crowded that I had to wait in line for almost an hour."),
 ("Mùa hè ở Tokyo nóng và ẩm hơn tôi nghĩ.", "東京の夏は思っていたより暑くて、湿気が多いです。", "Summer in Tokyo is hotter and more humid than I expected."),
 ("Bác sĩ khuyên tôi nên đi bộ mỗi ngày.", "医者に毎日歩くように勧められました。", "The doctor advised me to walk every day."),
 ("Chúc mừng sinh nhật, chúc bạn một năm thật nhiều sức khỏe.", "お誕生日おめでとうございます。健康で素敵な一年になりますように。", "Happy birthday, I wish you a year full of good health."),
 ("Cửa hàng này đang giảm giá 30% cho tất cả quần áo.", "この店では、すべての服が30%引きになっています。", "This store is offering 30% off all clothes."),
 ("Tôi quên mang ô nên bị ướt hết.", "傘を持ってくるのを忘れて、びしょ濡れになりました。", "I forgot my umbrella and got completely soaked."),
 ("Xe buýt sáng nay đến muộn gần 20 phút.", "今朝のバスは20分近く遅れて来ました。", "The bus was almost 20 minutes late this morning."),
 ("Bạn thích xem phim hành động hay phim hài hơn?", "アクション映画とコメディ映画、どちらが好きですか。", "Do you prefer action movies or comedies?"),
 ("Năm ngoái công ty tuyển thêm khoảng 50 nhân viên mới.", "去年、会社は新しい社員を50人ほど採用しました。", "Last year the company hired about 50 new employees."),
 ("Phòng nhân sự sẽ gửi thông báo về lịch nghỉ lễ.", "人事部から祝日の休みについてのお知らせが届きます。", "The HR department will send out a notice about the holiday schedule."),
 ("Doanh số tháng này thấp hơn dự kiến một chút.", "今月の売上は予想より少し低かったです。", "Sales this month were slightly lower than expected."),
 ("Khách hàng muốn ký hợp đồng trước cuối tháng.", "お客様は月末までに契約を結びたいそうです。", "The client wants to sign the contract before the end of the month."),
 ("Tôi sẽ đặt bàn ở nhà hàng cho buổi liên hoan tối thứ Sáu.", "金曜日の夜の飲み会のために、レストランを予約しておきます。", "I'll book a table at a restaurant for the party on Friday night."),
 ("Văn phòng mới rộng và sáng sủa hơn chỗ cũ nhiều.", "新しいオフィスは前の場所よりずっと広くて明るいです。", "The new office is much bigger and brighter than the old one."),
 ("Máy lạnh trong phòng họp hình như bị hỏng.", "会議室のエアコンが壊れているみたいです。", "The air conditioner in the meeting room seems to be broken."),
 ("Hôm qua tôi xem bóng đá đến tận nửa đêm.", "昨日は夜中までサッカーの試合を見ていました。", "Yesterday I watched football until midnight."),
 ("Con mèo nhà tôi rất thích ngủ trên bàn phím.", "うちの猫はキーボードの上で寝るのが大好きです。", "My cat loves sleeping on the keyboard."),
 ("Bạn có biết chỗ nào đổi tiền được tỷ giá tốt không?", "レートのいい両替所を知っていますか。", "Do you know anywhere to exchange money at a good rate?"),
 ("Tôi đang tiết kiệm tiền để mua nhà.", "家を買うためにお金を貯めています。", "I'm saving money to buy a house."),
 ("Trời nắng nóng thì nhớ uống nhiều nước nhé.", "暑い日は水をたくさん飲むようにしてくださいね。", "Remember to drink plenty of water when it's hot."),
 ("Kỳ nghỉ này tôi muốn đọc hết mấy cuốn sách đã mua.", "今度の休みに、買った本を全部読みたいです。", "This holiday I want to finish reading the books I bought."),
 ("Mẹ tôi nấu canh chua ngon nhất trên đời.", "母が作る酸っぱいスープは世界一おいしいです。", "My mom makes the best sour soup in the world."),
 ("Chuyến tàu cuối rời ga lúc 11 giờ đêm.", "最終電車は夜11時に駅を出ます。", "The last train leaves the station at 11 p.m."),
 ("Tuần sau tôi xin nghỉ 2 ngày để đi đám cưới bạn.", "来週、友達の結婚式に出るため2日間お休みをいただきます。", "Next week I'm taking 2 days off to go to a friend's wedding."),
 ("Dạo này rau ngoài chợ đắt hơn hẳn.", "最近、市場の野菜がずいぶん高くなりました。", "Vegetables at the market have become much more expensive lately."),
 ("Buổi sáng tôi thường chạy bộ quanh hồ.", "朝はたいてい湖の周りをジョギングしています。", "I usually go jogging around the lake in the morning."),
 ("Cảm ơn bạn đã ra sân bay đón tôi.", "空港まで迎えに来てくれてありがとうございます。", "Thank you for picking me up at the airport."),
 ("Xin lỗi, cho tôi hỏi nhà vệ sinh ở đâu?", "すみません、トイレはどこですか。", "Excuse me, where is the restroom?"),
 ("Thành phố đang xây thêm một tuyến tàu điện ngầm.", "市は地下鉄の新しい路線を建設しています。", "The city is building a new subway line."),
 ("Tôi bị dị ứng hải sản nên không ăn được món này.", "海鮮アレルギーがあるので、この料理は食べられません。", "I'm allergic to seafood, so I can't eat this dish."),
 ("Dịp lễ rất đông nên chúng ta nên đặt vé máy bay sớm.", "連休は混むので、早めに航空券を予約したほうがいいです。", "The holidays get very busy, so we should book our plane tickets early."),
 ("Công ty sẽ tổ chức du lịch cho nhân viên vào tháng 7.", "会社は7月に社員旅行を予定しています。", "The company is planning an employee trip in July."),
]
_seen = {r[l] for r in ALL for l in LANGS}
_dup = [x for g in GENERAL for x in g if x in _seen]
assert not _dup, f"Câu CHUNG trùng data: {_dup}"

ood_rows  = [{"vi": v, "ja": j, "en": e, "origin": "general_ood"} for v, j, e in GENERAL]
ood_clean = to_rows(ood_rows, None, levels=(0.0,))
ood_app   = to_rows(ood_rows, random.Random(7), levels=(1.0,))
score(ood_clean, "CHUNG sạch", app=True)
score(ood_app,   "CHUNG app",  app=True)
with model.disable_adapter():
    score(ood_clean, "CHUNG sạch", tag="gốc", app=True)
    score(ood_app,   "CHUNG app",  tag="gốc", app=True)

print("\n" + "=" * 72)
print(f"{'bộ':12s}{'chiều':8s}{'gốc':>8s}{'v9':>8s}{'chênh':>8s}   đánh giá")
print("=" * 72)
for label in ("CHUNG sạch", "CHUNG app"):
    for src, tgt in DIRECTIONS:
        g = RESULTS[("gốc", label, src, tgt)]["chrf"]
        v = RESULTS[("v9", label, src, tgt)]
        half = (v["ci"][1] - v["ci"][0]) / 2
        d = v["chrf"] - g
        verdict = "✅ không quên" if d >= -half else f"🔴 tụt quá CI — LoRA làm hỏng câu thường"
        print(f"{label:12s}{src}→{tgt:5s}{g:8.2f}{v['chrf']:8.2f}{d:+8.2f}   {verdict}")
print("\nv9 tụt so với gốc ở bộ CHUNG = đang học thuộc domain IT. Hạ lr / r, hoặc trộn thêm câu đời thường.")

In [ ]:
# CELL 10 — kiểm tra lỗi cụ thể: register vi→ja, sáng/chiều, con số
POLITE = re.compile(r"(です|ます|ません|ました|でした|ください|ましょう|ございます"
                    r"|ますか|ですか|ましたか|ませんか|ましょうか|でしょうか)[。！？]*$")

print("(1) REGISTER vi→ja — % output ở thể lịch sự (v6: 98.1% trên eval nhiễu)")
for label in ("SẠCH", "NHIỄU"):
    p = RESULTS[("v9", label, "vi", "ja")]["preds"]
    pol = 100 * sum(1 for x in p if POLITE.search(x.strip())) / len(p)
    print(f"  {label:6s} {pol:5.1f}%  {'✅' if pol >= 90 else '🔴'}")

_AM = {"en": r"\ba\.?m\.?\b|\bmorning\b", "ja": r"午前|朝", "vi": r"\bsáng\b"}
_PM = {"en": r"\bp\.?m\.?\b|\bafternoon\b|\bevening\b|\btonight\b",
       "ja": r"午後|夕方|夜", "vi": r"\bchiều\b|\btối\b"}
def ampm(t, lang):
    a = bool(re.search(_AM[lang], t, re.I)); p = bool(re.search(_PM[lang], t, re.I))
    return None if a == p else ("am" if a else "pm")

def numbers(t):
    t = unicodedata.normalize("NFKC", t)
    return {re.sub(r"[.,](?=\d{3}\b)", "", m) for m in re.findall(r"\d+(?:[.,]\d{3})*", t)}

print("\n(2) SÁNG/CHIỀU bị đảo và SỐ bị mất — trên eval sạch")
for src, tgt in DIRECTIONS:
    r = RESULTS[("v9", "SẠCH", src, tgt)]
    flip = sum(1 for p, ref in zip(r["preds"], r["refs"])
               if ampm(ref, tgt) and ampm(p, tgt) and ampm(ref, tgt) != ampm(p, tgt))
    lost = sum(1 for p, ref in zip(r["preds"], r["refs"])
               if numbers(ref) and not numbers(ref) <= numbers(p))
    print(f"  {src}→{tgt}: sáng/chiều đảo {flip} | mất/sai số {lost}")
print("  (eval 103 câu ít câu giờ giấc — contrast set ở CELL 11 mới đo trúng lỗi này)")

# (3) ĐẢO NGHĨA bật/tắt — lỗi nguy hiểm nhất v7 để lộ: マイクをオン -> "em tắt mic"
_ON  = {"vi": r"\b(bật|mở|kích hoạt|enable|cho phép)\b", "ja": r"(オン|有効|開け|許可)",
        "en": r"\b(turn(ed|s)? on|enable|open|allow|unmute)"}
_OFF = {"vi": r"\b(tắt|đóng|vô hiệu|disable|chặn)\b", "ja": r"(オフ|無効|閉じ|切っ|切ら|止め|停止)",
        "en": r"\b(turn(ed|s)? off|disable|close|block|mute)"}
def polarity(t, lang):
    on = bool(re.search(_ON[lang], t, re.I)); off = bool(re.search(_OFF[lang], t, re.I))
    return None if on == off else ("on" if on else "off")

print("\n(3) ĐẢO NGHĨA bật/tắt — trên eval sạch + nhiễu")
for src, tgt in DIRECTIONS:
    n = 0
    for label in ("SẠCH", "NHIỄU"):
        r = RESULTS[("v9", label, src, tgt)]
        n += sum(1 for p, ref in zip(r["preds"], r["refs"])
                 if polarity(ref, tgt) and polarity(p, tgt) and polarity(ref, tgt) != polarity(p, tgt))
    print(f"  {src}→{tgt}: {n} câu đảo nghĩa  {'✅' if n == 0 else '🔴'}")

In [ ]:
# CELL 11 — CONTRAST SET: 16 case v6 + 12 case tiếng Anh (v7) + 9 case v8 + 11 case v9
# (nguồn, src, tgt, PHẢI có, KHÔNG được có, mô tả)
# ⚠️ Câu nguồn KHÔNG được trùng câu nào trong data train — ô dưới tự kiểm tra.
CONTRAST = [
 # ---- v6 (giữ nguyên để so) ----
 ("Phần này ai rảnh nhận giúp không, em đang ôm hơi nhiều.", "vi", "ja", r"(手伝|助け)", r"抱擁", "ôm việc ≠ 抱擁"),
 ("Mọi người đồng ý thì mình tiến hành luôn nhé.", "vi", "ja", r"(ましょう)", r"てください", "rủ rê ≠ ra lệnh"),
 ("Cổng 8000 đang bị chiếm.", "vi", "ja", r"(使用中|使われて|使用され|埋まって)", r"占領", "port bận ≠ chiếm đóng"),
 ("Phân quyền chưa chặt, user thường vào được trang admin.", "vi", "ja", r"権限", r"特権", "権限 ≠ 特権"),
 ("Do đổi yêu cầu giữa chừng nên tiến độ chung bị ảnh hưởng.", "vi", "ja", r"(途中|変更)", r"中間要件", "đổi giữa chừng"),
 ("9 giờ sáng mai họp standup nhé mọi người.", "vi", "ja", r"9時", r"ごろ", "không thêm 'khoảng' vào giờ họp"),
 ("Mình bàn lại deadline vào chiều mai nhé.", "vi", "ja", r"(ましょう|ませんか)", r"しろ", "giữ thể lịch sự"),
 ("少し抱えすぎているので、余裕がある人に手伝ってほしいです。", "ja", "vi", r"(giúp|hỗ trợ)", r"muốn giúp đỡ", "~てほしい"),
 ("権限制御が甘く、一般ユーザーが管理画面に入れてしまいます。", "ja", "vi", r"(phân quyền|quyền)", r"mềm", "甘い = lỏng lẻo"),
 ("この件は明日までに対応していただけますか。", "ja", "vi", r"(giúp|nhờ|được không|nhé)", r"^Tôi sẽ", "nhờ vả ≠ tự nhận việc"),
 ("サーバーのメモリーが足りないので、スケールアップしてください。", "ja", "vi", r"(memory|bộ nhớ|RAM)", r"kỷ niệm", "メモリー = bộ nhớ"),
 ("レビューのコメントを反映しました。", "ja", "vi", r"(review|comment|nhận xét|góp ý)", r"bình luận phim", "review code"),
 ("恐れ入りますが、こちらの内容を来週までにご確認いただくことは可能でしょうか。", "ja", "vi",
  r"(giúp|anh|chị|được không|nhé|ạ)", r"^(Tôi|Em) sẽ", "kính ngữ lồng でしょうか"),
 ("Chị copy giúp em file config đó sang server mới với.", "vi", "ja", r"(コピー|していただけ)", r"手伝って", "giúp + động từ cụ thể"),
 ("Nếu cả nhóm không ai có ý kiến khác thì mình chốt kế hoạch này nhé.", "vi", "ja", r"(ましょう)", r"てください", "rủ rê tập thể"),
 ("Tuyệt đối không được tắt server trong giờ làm việc, nhớ kỹ điều này.", "vi", "ja",
  r"(ないでください|てください)", r"ましょう", "lệnh cấm ≠ rủ rê"),
 # ---- v7: tiếng Anh — nhắm đúng lỗi đo được trên model v6 ----
 ("Our call with the client is at 2 p.m. on Thursday.", "en", "vi", r"chiều", r"sáng", "p.m. = chiều"),
 ("Chiều nay 4 giờ mình họp lại nhé.", "vi", "en", r"(4 ?p\.?m|afternoon)", r"(\ba\.?m\b|morning)", "chiều = p.m."),
 ("9 giờ tối thứ Bảy mình bảo trì database.", "vi", "en", r"(9 ?p\.?m|night|evening)", r"(\b9 ?a\.?m|morning)", "tối = p.m."),
 ("Thanks for all your hard work this week, everyone.", "en", "vi", r"(cảm ơn|vất vả)", r"hạnh phúc", "cảm ơn cuối tuần"),
 ("Sorry for being late, my previous meeting ran over.", "en", "vi", r"xin lỗi", r"chạy", "ran over ≠ chạy"),
 ("I pushed the hotfix to the dev branch.", "en", "vi", r"push", r"(đẩy|ấn|nhấn)", "push giữ thuật ngữ"),
 ("The payment server has been down since this morning.", "en", "vi",
  r"(sập|down|ngừng|không hoạt động|lỗi)", r"giảm", "server down ≠ giảm"),
 ("The migration will cost around 1.5 million yen.", "en", "vi", r"1[.,]5 triệu", r"\b15 triệu", "1.5 million"),
 ("Chị xem giúp em cái log này với.", "vi", "en", r"(could you|can you|please)", r"see how", "nhờ xem giúp"),
 ("Chiều nay em hỏi lại team rồi phản hồi cho chị nhé.", "vi", "en",
  r"(get back|let you know|update you|reply|respond|tell you|answer)", r"\breturn", "phản hồi ≠ return"),
 ("Cổng 3000 đang bị process khác chiếm.", "vi", "en", r"(in use|used|taken|occupied|busy)", r"(seiz|captur|invad)", "port bận"),
 ("Nếu không ai phản đối thì mình merge luôn nhé.", "vi", "en", r"(let's|let us|we('ll| will| can)|go ahead)", r"^Please", "rủ rê ≠ ra lệnh"),
 # ---- v8: nhắm lỗi thấy ở output v7 (câu KHÁC data train v8) ----
 ("発言するときはマイクをオンにしてください。", "ja", "vi", r"bật", r"tắt", "オン = bật (v7 dịch thành tắt)"),
 ("Tắt thông báo email cho tài khoản test giúp em.", "vi", "ja", r"(オフ|無効|停止|止め)", r"(オン|有効に)", "tắt = オフ"),
 ("Nhớ bật lại firewall sau khi test xong nhé.", "vi", "en", r"(turn .*on|enable|re-?enable|switch .*on)", r"(turn .*off|disable)", "bật lại = turn back on"),
 ("Please disable the old API key after the migration.", "en", "vi", r"(tắt|vô hiệu|disable|hủy|huỷ|xoá|xóa|thu hồi)", r"(bật|kích hoạt|enable)", "disable ≠ bật"),
 ("Could you rename this variable?", "en", "vi", r"biến", r"tên thay đổi", "variable = biến"),
 ("Let's push the release to next Friday.", "en", "vi", r"(dời|lùi|hoãn|chuyển)", r"push", "push lịch = dời"),
 ("The server crashed, so the job didn't finish.", "en", "vi", r"(nên|vì|do)", r"để (khỏi|không)", "so = nên (nhân quả)"),
 ("Anh ấy đang ôm nhiều dự án quá.", "vi", "ja", r"抱え", r"(抱擁|抱きし)", "ôm dự án = 抱える"),
 ("Cổng 9000 đang bị chiếm, em đổi cổng khác nhé.", "vi", "ja", r"(使用中|使われて|使用され|埋まって|占有)", r"占領", "cổng bị chiếm ≠ 占領"),
 # ---- v9: đúng dạng backend đưa vào model (placeholder tên, số bằng chữ, câu ngắn, thuật ngữ BrSE)
 ("PnA が明日のデモを担当します", "ja", "vi", r"PnA", r"(?!)", "placeholder tên giữ nguyên"),
 ("この件は PnA と PnB に確認してください", "ja", "vi", r"PnA.*PnB|PnB.*PnA", r"(?!)", "2 placeholder"),
 ("chiba-san hôm nay có tham gia họp không", "vi", "ja", r"Chiba", r"(千|1[.,]?000)", "tên Latin chép nguyên"),
 ("hẹn ba giờ chiều mai nhé", "vi", "en", r"(\b3\b|three)", r"(\ba\.?m\b|morning)", "số bằng chữ từ ASR"),
 ("chi phí tăng mười lăm phần trăm so với quý trước", "vi", "ja", r"15", r"(?!)", "mười lăm phần trăm = 15%"),
 ("dạ vâng", "vi", "ja", r"はい", r"(おめでとう|ありがとう)", "câu ngắn không bịa nghĩa"),
 ("この機能の工数を見積もっていただけますか。", "ja", "vi", r"(effort|công|thời gian|ước tính|estimate)", r"công nhân", "工数"),
 ("まず原因を切り分けてから対応しましょう。", "ja", "vi", r"(khoanh vùng|xác định|tìm|cô lập|phân tách|tách)", r"\bcắt\b", "切り分け ≠ cắt"),
 ("この対応を他の画面にも横展開してください。", "ja", "vi", r"(áp dụng|màn hình khác)", r"ngang", "横展開 ≠ ngang"),
 ("一度持ち帰って検討させてください。", "ja", "vi", r"(xem xét|trao đổi|thảo luận|cân nhắc)", r"mang về nhà", "持ち帰り = mang về bàn"),
 ("chị xem giúp em cái lỗi này với", "vi", "en", r"(could you|can you|please)", r"\b(he|him|brother|sister|older)\b", "chị = người nghe (you)"),
]

train_src = {r[l] for r in train_rows for l in LANGS}
dup = [c[0] for c in CONTRAST if c[0] in train_src]
assert not dup, f"Contrast trùng data train (chỉ đo trí nhớ): {dup}"

def run_contrast(tag):
    passed, lines = 0, []
    for srct, s, t, must, mustnt, desc in CONTRAST:
        out = gen_batch([srct], s, t)[0]
        ok = bool(re.search(must, out, re.I)) and not re.search(mustnt, out, re.I)
        passed += ok
        lines.append((ok, s, t, desc, srct, out))
    return passed, lines

p7, lines7 = run_contrast("v9")
with model.disable_adapter():
    pg, _ = run_contrast("gốc")

for ok, s, t, desc, srct, out in lines7:
    print(f"  {'✅' if ok else '🔴'} [{s}→{t}] {desc}")
    if not ok:
        print(f"       nguồn: {srct}\n       model: {out}")
groups = [("v6", 0, 16, "v6: 14/16, v7: 14/16"), ("v7-en", 16, 28, "v7: 12/12"),
          ("v8", 28, 37, "xem lần chạy v8"), ("v9", 37, len(CONTRAST), "mới")]
print(f"\n  v9 ĐẠT {p7}/{len(CONTRAST)}  (M2M-100 gốc: {pg}/{len(CONTRAST)})")
for name, a, b, ref in groups:
    print(f"    phần {name:6s}: {sum(ok for ok, *_ in lines7[a:b])}/{b - a}   (mốc {ref})")

In [ ]:
# CELL 12 — soi nhanh: câu tệ nhất mỗi chiều + dịch thử kiểu ASR
print("=" * 72); print("3 CÂU TỆ NHẤT MỖI CHIỀU (eval nhiễu)"); print("=" * 72)
for src, tgt in DIRECTIONS:
    r = RESULTS[("v9", "NHIỄU", src, tgt)]
    worst = sorted(zip(r["sent"], r["srcs"], r["refs"], r["preds"]))[:3]
    for sc, a, b, c in worst:
        print(f"[{src}→{tgt}] chrF={sc:.1f}\n  nguồn: {a}\n  chuẩn: {b}\n  model: {c}")

@torch.no_grad()
def dich(text, src, tgt):
    return gen_batch([text], src, tgt)[0]

print("\n" + "=" * 72); print("DỊCH THỬ"); print("=" * 72)
for t, s, g in [("The meeting starts at 3 p.m. tomorrow.", "en", "vi"),
                ("Can we move the review to 4.30 in the afternoon", "en", "vi"),
                ("I'll check it and get back to you later.", "en", "vi"),
                ("anh xem giúp em file này nhé", "vi", "en"),
                ("cảm ơn mọi người hôm nay vất vả rồi", "vi", "en"),
                ("明日の会議は午後3時からです", "ja", "vi"),
                ("em gửi tài liệu rồi anh xem giúp em nhé", "vi", "ja"),
                ("カメラをオフにして、マイクだけオンにしてください", "ja", "vi"),
                ("It's fine to move this task to the next sprint.", "en", "vi"),
                ("tắt chế độ debug rồi mới deploy nhé", "vi", "en"),
                ("PnA から差分をもらったら本番反映します", "ja", "vi"),
                ("hai mươi người đã tham gia buổi demo", "vi", "en"),
                ("dạ em hiểu rồi em sẽ sửa ngay", "vi", "ja"),
                ("影響範囲を確認してから横展開しましょう", "ja", "vi"),
                ("em ước tính khoảng ba man day", "vi", "ja")]:
    print(f"  [{s}→{g}] {t}\n          → {dich(t, s, g)}")

In [ ]:
# CELL 13 — merge LoRA vào model gốc
merged = model.merge_and_unload()        # sau lệnh này `model` không dùng lại được
merged.save_pretrained("m2m418-it-merged-v9")
tokenizer.save_pretrained("m2m418-it-merged-v9")
for f in sorted(os.listdir("m2m418-it-merged-v9")):
    print(f"  {f:32s} {os.path.getsize(f'm2m418-it-merged-v9/{f}')/1e6:8.2f} MB")

In [ ]:
# CELL 14 — convert sang CTranslate2 int8 (định dạng backend đang dùng)
!pip install -q ctranslate2
import subprocess

SRC, OUT = "m2m418-it-merged-v9", "m2m100_418M_int8_v9"
CANDIDATES = ["sentencepiece.bpe.model", "vocab.json", "tokenizer_config.json",
              "special_tokens_map.json", "added_tokens.json", "tokenizer.json"]
present = [f for f in CANDIDATES if os.path.exists(f"{SRC}/{f}")]
print("copy :", present)
if "tokenizer_config.json" not in present or "sentencepiece.bpe.model" not in present:
    raise RuntimeError("Thiếu file tokenizer — backend sẽ không nạp được (chạy lại CELL 13).")
subprocess.run(["ct2-transformers-converter", "--model", SRC, "--output_dir", OUT,
                "--quantization", "int8", "--force", "--copy_files", *present], check=True)
print("✅ Convert xong")

In [ ]:
# CELL 15 — test bản CT2 trên CPU (y như backend) + đóng gói
import ctranslate2
from transformers import AutoTokenizer

lack = [f for f in ["model.bin", "config.json", "shared_vocabulary.json"] if not os.path.exists(f"{OUT}/{f}")]
if lack:
    raise RuntimeError(f"Thiếu file lõi {lack} — convert hỏng.")

tok = AutoTokenizer.from_pretrained(OUT, local_files_only=True)
ct2 = ctranslate2.Translator(OUT, device="cpu", compute_type="int8")
LEN_MULT = {("ja", "vi"): 4.0, ("vi", "ja"): 2.0, ("en", "vi"): 3.0, ("vi", "en"): 2.0}   # = backend/main.py

def dich_ct2(text, src, tgt):
    tok.src_lang = src
    st = tok.convert_ids_to_tokens(tok.encode(text.lower() if src == "vi" else text))
    res = ct2.translate_batch([st], target_prefix=[[f"__{tgt}__"]], beam_size=4,
                              max_decoding_length=min(128, int(len(st) * LEN_MULT[(src, tgt)]) + 12),
                              repetition_penalty=1.1, no_repeat_ngram_size=3)
    return tok.decode(tok.convert_tokens_to_ids(res[0].hypotheses[0][1:]), skip_special_tokens=True).strip()

print("--- CT2 INT8 trên CPU ---")
for t, s, g in [("この部分は納品前に再確認が必要です", "ja", "vi"),
                ("cổng 8000 đang bị chiếm", "vi", "ja"),
                ("The staging server has been down since 9 a.m.", "en", "vi"),
                ("chiều nay 3 giờ mình review code nhé", "vi", "en"),
                ("PnA が資料を送りました", "ja", "vi"),
                ("mười lăm phần trăm người dùng bị lỗi", "vi", "ja")]:
    print(f"  [{s}→{g}] {t}\n            → {dich_ct2(t, s, g)}")

!zip -qr {OUT}.zip {OUT}
print(f"\n📦 {OUT}.zip  {os.path.getsize(f'{OUT}.zip')/1e6:.1f} MB — tải về từ panel Files bên trái.")
# Hoặc chép sang Drive:
# import shutil; shutil.copy(f"{OUT}.zip", "/content/drive/MyDrive/")

## Bước tiếp theo ở máy local (A/B với bản đang chạy)

1. Giải nén zip vào `backend/models/m2m100_418M_int8_v9/` (**không** ghi đè `m2m100_418M_int8/`).
2. Đo cùng một bộ câu cho bản đang chạy và v9 (PowerShell, trong `backend/`, venv đã activate):
   ```powershell
   python eval_mt.py --out eval_cur.csv
   $env:KATOBA_MT_DIR = "models\m2m100_418M_int8_v9"; python eval_mt.py --out eval_v9.csv; Remove-Item Env:KATOBA_MT_DIR
   ```
   `eval_mt.py` chạy đúng pipeline của app (hậu xử lý + glossary).
   ⚠️ `testdata/cau_test_mt.tsv` có vài câu cùng chủ đề với data targeted nên số sẽ hơi lạc quan — CELL 9 mới là mốc quyết định.
3. Chạy thử app thật với `$env:KATOBA_MT_DIR` như trên — nhất là họp có tên người Nhật chưa khai trong people.json
   (xem log `M2M-100 nuốt mất ... tên người` có giảm không) và nói số bằng miệng tiếng Việt.
4. Tốt hơn thật → đổi tên thư mục thành `m2m100_418M_int8`, sửa `MODEL_VERSION = "v9"` trong `main.py`.